In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import random
import shutil
import sys
import tarfile
from collections import defaultdict
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content


In [ ]:
%pip install -q xarray netCDF4 pandas pyarrow


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

TORNET_ARCHIVE_DIR = Path("/content/drive/MyDrive/TorNet_Backup")

expected_archives = {
    "tornet_2013.tar.gz": 2.94,
    "tornet_2014.tar.gz": 14.03,
    "tornet_2015.tar.gz": 16.20,
    "tornet_2016.tar.gz": 15.13,
    "tornet_2017.tar.gz": 14.08,
    "tornet_2018.tar.gz": 11.65,
    "tornet_2019.tar.gz": 16.97,
    "tornet_2020.tar.gz": 15.86,
    "tornet_2021.tar.gz": 17.07,
    "tornet_2022.tar.gz": 17.72,
}

rows = []

for filename, expected_size_gib in expected_archives.items():
    path = TORNET_ARCHIVE_DIR / filename
    exists = path.is_file()
    actual_size_gib = path.stat().st_size / (1024**3) if exists else None

    rows.append(
        {
            "filename": filename,
            "exists": exists,
            "expected_size_gib": expected_size_gib,
            "actual_size_gib": actual_size_gib,
            "size_difference_gib": (
                actual_size_gib - expected_size_gib
                if actual_size_gib is not None
                else None
            ),
        }
    )

archive_inventory = pd.DataFrame(rows)
archive_inventory

,filename,exists,expected_size_gib,actual_size_gib,size_difference_gib
0,tornet_2013.tar.gz,True,2.94,2.942855,0.002855
1,tornet_2014.tar.gz,True,14.03,14.031784,0.001784
2,tornet_2015.tar.gz,True,16.20,16.200937,0.000937
3,tornet_2016.tar.gz,True,15.13,15.129823,-0.000177
4,tornet_2017.tar.gz,True,14.08,14.078567,-0.001433
5,tornet_2018.tar.gz,True,11.65,11.651212,0.001212
6,tornet_2019.tar.gz,True,16.97,16.968593,-0.001407
7,tornet_2020.tar.gz,True,15.86,15.858887,-0.001113
8,tornet_2021.tar.gz,True,17.07,17.072932,0.002932
9,tornet_2022.tar.gz,True,17.72,17.717185,-0.002815


In [5]:
missing = archive_inventory.loc[~archive_inventory["exists"], "filename"].tolist()

if missing:
    raise FileNotFoundError(f"Missing TorNet archives: {missing}")

print(
    f"Found {len(archive_inventory)} archives totaling "
    f"{archive_inventory['actual_size_gib'].sum():.2f} GiB"
)

Found 10 archives totaling 141.65 GiB


In [6]:
SAMPLE_YEAR = "2013"
SAMPLE_ARCHIVE = TORNET_ARCHIVE_DIR / f"tornet_{SAMPLE_YEAR}.tar.gz"
SAMPLE_OUTPUT_DIR = Path("/content/tornet_audit_samples") / SAMPLE_YEAR

EXPECTED_SPLITS = {"train", "test"}
EXPECTED_CATEGORIES = {"NUL", "WRN", "TOR"}

if not SAMPLE_ARCHIVE.is_file():
    raise FileNotFoundError(f"Archive not found: {SAMPLE_ARCHIVE}")

# This directory is runtime-local and contains only small audit samples.
# Clear it so stale files from earlier executions cannot contaminate the audit.
if SAMPLE_OUTPUT_DIR.exists():
    shutil.rmtree(SAMPLE_OUTPUT_DIR)

SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def parse_tornet_member(
    member_name: str,
    *,
    expected_year: str,
) -> tuple[str, str, str] | None:
    """Parse split, year, and category from a TorNet NetCDF member path."""

    parts = tuple(
        part
        for part in PurePosixPath(member_name).parts
        if part not in {"", "."}
    )

    # Observed TorNet layout:
    # train/2013/NUL_*.nc
    # test/2013/TOR_*.nc
    if len(parts) != 3:
        return None

    split, year, filename = parts

    if split not in EXPECTED_SPLITS:
        return None

    if year != expected_year:
        return None

    if not filename.lower().endswith(".nc"):
        return None

    if "_" not in filename:
        return None

    category = filename.split("_", maxsplit=1)[0].upper()

    if not category:
        return None

    return split, year, category


def copy_member_with_sha256(
    source,
    destination: Path,
    *,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Copy one archive member and return its SHA-256 digest."""

    digest = hashlib.sha256()

    with destination.open("wb") as output:
        while True:
            chunk = source.read(chunk_size)

            if not chunk:
                break

            output.write(chunk)
            digest.update(chunk)

    return digest.hexdigest()


combination_counts: dict[tuple[str, str], int] = defaultdict(int)
selected_by_combination: dict[tuple[str, str], dict] = {}

netcdf_member_count = 0
unparsed_netcdf_member_count = 0
unparsed_netcdf_examples: list[str] = []

with tarfile.open(SAMPLE_ARCHIVE, mode="r|gz") as archive:
    for archive_index, member in enumerate(archive):
        if not member.isfile():
            continue

        if not member.name.lower().endswith(".nc"):
            continue

        netcdf_member_count += 1

        parsed = parse_tornet_member(
            member.name,
            expected_year=SAMPLE_YEAR,
        )

        if parsed is None:
            unparsed_netcdf_member_count += 1

            if len(unparsed_netcdf_examples) < 20:
                unparsed_netcdf_examples.append(member.name)

            continue

        split, year, category = parsed
        combination = (split, category)
        combination_counts[combination] += 1

        # The first file encountered for each combination is deterministic
        # because tar archive member order is fixed.
        if combination in selected_by_combination:
            continue

        source = archive.extractfile(member)

        if source is None:
            raise RuntimeError(
                f"Could not open selected archive member: {member.name}"
            )

        filename = PurePosixPath(member.name).name
        destination = (
            SAMPLE_OUTPUT_DIR
            / f"{split}__{category}__{filename}"
        )

        with source:
            sha256 = copy_member_with_sha256(source, destination)

        extracted_size = destination.stat().st_size

        if extracted_size != member.size:
            raise IOError(
                "Extracted sample size mismatch for "
                f"{member.name}: expected {member.size}, "
                f"found {extracted_size}"
            )

        selected_by_combination[combination] = {
            "split": split,
            "year": year,
            "category": category,
            "archive_index": archive_index,
            "archive_member": member.name,
            "size_bytes": member.size,
            "sha256": sha256,
            "local_sample_path": str(destination),
        }

        print(
            "Selected "
            f"{split}/{category}: {member.name}"
        )

        if netcdf_member_count % 1_000 == 0:
            print(
                f"Scanned {netcdf_member_count:,} NetCDF members; "
                f"found {len(selected_by_combination)} combinations"
            )


if not combination_counts:
    raise RuntimeError(
        f"No valid TorNet NetCDF members found in {SAMPLE_ARCHIVE}"
    )

combination_count_df = (
    pd.DataFrame(
        [
            {
                "split": split,
                "category": category,
                "file_count": count,
            }
            for (split, category), count in combination_counts.items()
        ]
    )
    .sort_values(["split", "category"])
    .reset_index(drop=True)
)

selected_sample_df = (
    pd.DataFrame(selected_by_combination.values())
    .sort_values(["split", "category"])
    .reset_index(drop=True)
)

available_combinations = set(combination_counts)
selected_combinations = set(selected_by_combination)

if selected_combinations != available_combinations:
    raise AssertionError(
        "Selected combinations do not match available combinations: "
        f"available={sorted(available_combinations)}, "
        f"selected={sorted(selected_combinations)}"
    )

if selected_sample_df.duplicated(["split", "category"]).any():
    raise AssertionError(
        "More than one sample was selected for a split/category combination"
    )

unexpected_categories = (
    set(combination_count_df["category"]) - EXPECTED_CATEGORIES
)

missing_expected_combinations = (
    {
        (split, category)
        for split in EXPECTED_SPLITS
        for category in EXPECTED_CATEGORIES
    }
    - available_combinations
)

print()
print(f"NetCDF members scanned: {netcdf_member_count:,}")
print(
    "Available split/category combinations:",
    sorted(available_combinations),
)
print(
    "Unexpected categories:",
    sorted(unexpected_categories) or "None",
)
print(
    "Expected combinations absent from 2013:",
    sorted(missing_expected_combinations) or "None",
)
print(
    "Unparsed NetCDF members:",
    f"{unparsed_netcdf_member_count:,}",
)

if unparsed_netcdf_examples:
    print("First unparsed examples:")
    for member_name in unparsed_netcdf_examples:
        print(" -", member_name)

display(combination_count_df)
display(selected_sample_df)

Selected train/NUL: train/2013/NUL_131101_063025_KRLX_476088s_F5.nc
Selected train/WRN: train/2013/WRN_130908_005945_KUDX_1073301n_E2.nc
Selected train/TOR: train/2013/TOR_131031_150019_KLCH_480352_W9.nc
Selected test/WRN: test/2013/WRN_131005_223742_KPAH_1073330n_H9.nc
Selected test/NUL: test/2013/NUL_130916_011006_KPDT_472604s_D4.nc
Selected test/TOR: test/2013/TOR_131004_040911_KOAX_472700_A1.nc

NetCDF members scanned: 4,071
Available split/category combinations: [('test', 'NUL'), ('test', 'TOR'), ('test', 'WRN'), ('train', 'NUL'), ('train', 'TOR'), ('train', 'WRN')]
Unexpected categories: None
Expected combinations absent from 2013: None
Unparsed NetCDF members: 0


,split,category,file_count
0,test,NUL,253
1,test,TOR,60
2,test,WRN,260
3,train,NUL,2548
4,train,TOR,329
5,train,WRN,621


,split,year,category,archive_index,archive_member,size_bytes,sha256,local_sample_path
0,test,2013,NUL,3502,test/2013/NUL_130916_011006_KPDT_472604s_D4.nc,993660,80dc82ef306abaa3ab4d492f17d07f6d10e34727c2589b...,/content/tornet_audit_samples/2013/test__NUL__...
1,test,2013,TOR,3508,test/2013/TOR_131004_040911_KOAX_472700_A1.nc,1065616,4939be7fc2a1d07847a332e32739c10fbc0500d6020ae9...,/content/tornet_audit_samples/2013/test__TOR__...
2,test,2013,WRN,3501,test/2013/WRN_131005_223742_KPAH_1073330n_H9.nc,908353,16a203223d929f69647aaacb03eb0d5a0706c8907543de...,/content/tornet_audit_samples/2013/test__WRN__...
3,train,2013,NUL,2,train/2013/NUL_131101_063025_KRLX_476088s_F5.nc,1010954,d9b389ca9b3dc6d111c977807491ac7d5fe161ab90754b...,/content/tornet_audit_samples/2013/train__NUL_...
4,train,2013,TOR,23,train/2013/TOR_131031_150019_KLCH_480352_W9.nc,1047881,57236e1b75416e47c28ea0c682e3bd8b2705a239db2bec...,/content/tornet_audit_samples/2013/train__TOR_...
5,train,2013,WRN,8,train/2013/WRN_130908_005945_KUDX_1073301n_E2.nc,851808,b3c9e0cf68849493b05e548a55b9a12ba43123201e7006...,/content/tornet_audit_samples/2013/train__WRN_...


In [7]:
import xarray as xr
from netCDF4 import Dataset as NetCDFDataset


def metadata_to_jsonable(value):
    """Convert NetCDF/xarray metadata into stable JSON-compatible values."""

    if value is None or isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, float):
        if np.isfinite(value):
            return value

        return str(value)

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, np.generic):
        return metadata_to_jsonable(value.item())

    if isinstance(value, np.dtype):
        return str(value)

    if isinstance(value, np.ndarray):
        return [
            metadata_to_jsonable(item)
            for item in value.tolist()
        ]

    if isinstance(value, (list, tuple, set, frozenset)):
        return [
            metadata_to_jsonable(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(key): metadata_to_jsonable(item)
            for key, item in value.items()
        }

    if isinstance(value, Path):
        return str(value)

    return str(value)


def capture_netcdf_metadata(callable_):
    """Capture optional native NetCDF metadata without hiding failures."""

    try:
        return metadata_to_jsonable(callable_())
    except Exception as exc:
        return {
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


dataset_metadata_records: list[dict] = []
variable_metadata_records: list[dict] = []

for sample in selected_sample_df.itertuples(index=False):
    sample_path = Path(sample.local_sample_path)

    if not sample_path.is_file():
        raise FileNotFoundError(
            f"Selected sample no longer exists: {sample_path}"
        )

    with (
        xr.open_dataset(
            sample_path,
            engine="netcdf4",
            decode_cf=True,
            mask_and_scale=True,
            decode_times=True,
        ) as decoded_dataset,
        xr.open_dataset(
            sample_path,
            engine="netcdf4",
            decode_cf=False,
            mask_and_scale=False,
            decode_times=False,
        ) as raw_dataset,
        NetCDFDataset(sample_path, mode="r") as native_dataset,
    ):
        decoded_category = str(
            decoded_dataset.attrs.get("category", "")
        ).upper()

        if decoded_category != sample.category:
            raise ValueError(
                "Category mismatch between archive path and dataset "
                f"attributes for {sample.archive_member}: "
                f"path={sample.category!r}, "
                f"attribute={decoded_category!r}"
            )

        decoded_variable_names = set(decoded_dataset.variables)
        raw_variable_names = set(raw_dataset.variables)
        native_variable_names = set(native_dataset.variables)

        if not (
            decoded_variable_names
            == raw_variable_names
            == native_variable_names
        ):
            raise ValueError(
                "Variable-name mismatch across decoded, raw, and native "
                f"views for {sample.archive_member}"
            )

        dataset_metadata_records.append(
            {
                "split": sample.split,
                "year": sample.year,
                "category": sample.category,
                "archive_index": sample.archive_index,
                "archive_member": sample.archive_member,
                "sample_sha256": sample.sha256,
                "sample_size_bytes": sample.size_bytes,
                "netcdf_file_format": native_dataset.file_format,
                "netcdf_data_model": native_dataset.data_model,
                "decoded_dimensions": metadata_to_jsonable(
                    dict(decoded_dataset.sizes)
                ),
                "decoded_data_variables": sorted(
                    decoded_dataset.data_vars
                ),
                "decoded_coordinates": sorted(decoded_dataset.coords),
                "decoded_global_attributes": metadata_to_jsonable(
                    dict(decoded_dataset.attrs)
                ),
                "decoded_dataset_encoding": metadata_to_jsonable(
                    dict(decoded_dataset.encoding)
                ),
                "raw_dimensions": metadata_to_jsonable(
                    dict(raw_dataset.sizes)
                ),
                "raw_global_attributes": metadata_to_jsonable(
                    dict(raw_dataset.attrs)
                ),
                "raw_dataset_encoding": metadata_to_jsonable(
                    dict(raw_dataset.encoding)
                ),
                "native_global_attributes": metadata_to_jsonable(
                    {
                        attribute_name: native_dataset.getncattr(
                            attribute_name
                        )
                        for attribute_name in native_dataset.ncattrs()
                    }
                ),
            }
        )

        for variable_name in sorted(decoded_dataset.variables):
            decoded_variable = decoded_dataset[variable_name]
            raw_variable = raw_dataset[variable_name]
            native_variable = native_dataset.variables[variable_name]

            decoded_values = np.asarray(decoded_variable.values)

            finite_fraction = None
            minimum = None
            maximum = None

            if (
                decoded_values.size > 0
                and np.issubdtype(decoded_values.dtype, np.number)
            ):
                finite_mask = np.isfinite(decoded_values)
                finite_fraction = float(finite_mask.mean())

                if finite_mask.any():
                    finite_values = decoded_values[finite_mask]
                    minimum = float(finite_values.min())
                    maximum = float(finite_values.max())

            variable_metadata_records.append(
                {
                    "split": sample.split,
                    "category": sample.category,
                    "archive_member": sample.archive_member,
                    "variable": variable_name,
                    "decoded_dimensions": tuple(decoded_variable.dims),
                    "decoded_shape": tuple(decoded_variable.shape),
                    "decoded_dtype": str(decoded_variable.dtype),
                    "decoded_finite_fraction": finite_fraction,
                    "decoded_minimum": minimum,
                    "decoded_maximum": maximum,
                    "decoded_attributes": metadata_to_jsonable(
                        dict(decoded_variable.attrs)
                    ),
                    "decoded_xarray_encoding": metadata_to_jsonable(
                        dict(decoded_variable.encoding)
                    ),
                    "raw_dimensions": tuple(raw_variable.dims),
                    "raw_shape": tuple(raw_variable.shape),
                    "raw_dtype": str(raw_variable.dtype),
                    "raw_attributes": metadata_to_jsonable(
                        dict(raw_variable.attrs)
                    ),
                    "raw_xarray_encoding": metadata_to_jsonable(
                        dict(raw_variable.encoding)
                    ),
                    "native_dimensions": tuple(
                        native_variable.dimensions
                    ),
                    "native_shape": tuple(native_variable.shape),
                    "native_datatype": str(native_variable.datatype),
                    "native_attributes": metadata_to_jsonable(
                        {
                            attribute_name: native_variable.getncattr(
                                attribute_name
                            )
                            for attribute_name in native_variable.ncattrs()
                        }
                    ),
                    "native_chunking": capture_netcdf_metadata(
                        native_variable.chunking
                    ),
                    "native_filters": capture_netcdf_metadata(
                        native_variable.filters
                    ),
                    "native_endianness": capture_netcdf_metadata(
                        native_variable.endian
                    ),
                }
            )

dataset_metadata_df = pd.DataFrame(dataset_metadata_records)
variable_metadata_df = pd.DataFrame(variable_metadata_records)

if len(dataset_metadata_df) != len(selected_sample_df):
    raise AssertionError(
        "Dataset metadata record count does not match selected sample count"
    )

print(
    f"Captured dataset metadata for "
    f"{len(dataset_metadata_df)} representative files"
)
print(
    f"Captured variable metadata for "
    f"{len(variable_metadata_df)} file-variable pairs"
)

Captured dataset metadata for 6 representative files
Captured variable metadata for 90 file-variable pairs


In [8]:
dataset_summary_columns = [
    "split",
    "category",
    "archive_member",
    "sample_sha256",
    "netcdf_file_format",
    "netcdf_data_model",
    "decoded_dimensions",
    "decoded_data_variables",
    "decoded_coordinates",
]

display(
    dataset_metadata_df[
        dataset_summary_columns
    ].sort_values(["split", "category"]).reset_index(drop=True)
)

PRIMARY_RADAR_VARIABLES = [
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
]

AUDITED_MODEL_VARIABLES = [
    *PRIMARY_RADAR_VARIABLES,
    "frame_labels",
    "range_folded_mask",
]


def stable_unique(values) -> list[str]:
    """Return deterministic JSON representations of unique metadata values."""

    serialized_values = {
        json.dumps(
            metadata_to_jsonable(value),
            sort_keys=True,
        )
        for value in values
    }

    return sorted(serialized_values)


encoding_summary_rows = []

for variable_name in AUDITED_MODEL_VARIABLES:
    records = [
        record
        for record in variable_metadata_records
        if record["variable"] == variable_name
    ]

    if len(records) != len(selected_sample_df):
        raise AssertionError(
            f"Expected {len(selected_sample_df)} representative records "
            f"for {variable_name}; found {len(records)}"
        )

    encoding_summary_rows.append(
        {
            "variable": variable_name,
            "representative_files": len(records),
            "decoded_shapes": stable_unique(
                record["decoded_shape"] for record in records
            ),
            "decoded_dtypes": stable_unique(
                record["decoded_dtype"] for record in records
            ),
            "raw_dtypes": stable_unique(
                record["raw_dtype"] for record in records
            ),
            "fill_values": stable_unique(
                record["raw_attributes"].get("_FillValue")
                for record in records
            ),
            "least_significant_digits": stable_unique(
                record["raw_xarray_encoding"].get(
                    "least_significant_digit"
                )
                for record in records
            ),
            "native_chunking": stable_unique(
                record["native_chunking"] for record in records
            ),
            "native_filters": stable_unique(
                record["native_filters"] for record in records
            ),
            "native_endianness": stable_unique(
                record["native_endianness"] for record in records
            ),
        }
    )

encoding_summary_df = pd.DataFrame(encoding_summary_rows)
display(encoding_summary_df)

missingness_df = (
    variable_metadata_df.loc[
        variable_metadata_df["variable"].isin(
            PRIMARY_RADAR_VARIABLES
        ),
        [
            "split",
            "category",
            "variable",
            "decoded_finite_fraction",
        ],
    ]
    .pivot(
        index=["split", "category"],
        columns="variable",
        values="decoded_finite_fraction",
    )
    .reset_index()
    .sort_values(["split", "category"])
    .reset_index(drop=True)
)

display(missingness_df)

frame_label_records = []

for sample in (
    selected_sample_df
    .sort_values(["split", "category"])
    .itertuples(index=False)
):
    sample_path = Path(sample.local_sample_path)

    with xr.open_dataset(
        sample_path,
        engine="netcdf4",
        decode_cf=True,
        mask_and_scale=True,
        decode_times=True,
    ) as dataset:
        frame_labels = np.asarray(
            dataset["frame_labels"].values
        ).astype(np.int64)

        frame_times = np.asarray(dataset["time"].values)

        if frame_labels.ndim != 1:
            raise AssertionError(
                f"Expected one-dimensional frame labels for "
                f"{sample.archive_member}; found {frame_labels.shape}"
            )

        if frame_labels.shape != frame_times.shape:
            raise AssertionError(
                f"Frame-label/time shape mismatch for "
                f"{sample.archive_member}: "
                f"labels={frame_labels.shape}, times={frame_times.shape}"
            )

        unique_labels = set(np.unique(frame_labels).tolist())

        if not unique_labels.issubset({0, 1}):
            raise AssertionError(
                f"Unexpected frame-label values for "
                f"{sample.archive_member}: {sorted(unique_labels)}"
            )

        frame_label_records.append(
            {
                "split": sample.split,
                "category": sample.category,
                "archive_member": sample.archive_member,
                "event_id": dataset.attrs.get("event_id"),
                "episode_id": dataset.attrs.get("episode_id"),
                "ef_number": dataset.attrs.get("ef_number"),
                "tornado_start_time": dataset.attrs.get(
                    "tornado_start_time"
                ),
                "tornado_end_time": dataset.attrs.get(
                    "tornado_end_time"
                ),
                "frame_count": len(frame_labels),
                "positive_frame_count": int(frame_labels.sum()),
                "frame_labels": frame_labels.tolist(),
                "frame_times": [
                    pd.Timestamp(value).isoformat()
                    for value in frame_times
                ],
            }
        )

frame_label_summary_df = pd.DataFrame(frame_label_records)

display(
    frame_label_summary_df.sort_values(
        ["split", "category"]
    ).reset_index(drop=True)
)

print(
    "Concise audit complete: "
    f"{len(dataset_metadata_df)} representative files, "
    f"{len(variable_metadata_df)} file-variable records, "
    f"{len(frame_label_summary_df)} frame-label summaries."
)


,split,category,archive_member,sample_sha256,netcdf_file_format,netcdf_data_model,decoded_dimensions,decoded_data_variables,decoded_coordinates
0,test,NUL,test/2013/NUL_130916_011006_KPDT_472604s_D4.nc,80dc82ef306abaa3ab4d492f17d07f6d10e34727c2589b...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"
1,test,TOR,test/2013/TOR_131004_040911_KOAX_472700_A1.nc,4939be7fc2a1d07847a332e32739c10fbc0500d6020ae9...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"
2,test,WRN,test/2013/WRN_131005_223742_KPAH_1073330n_H9.nc,16a203223d929f69647aaacb03eb0d5a0706c8907543de...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"
3,train,NUL,train/2013/NUL_131101_063025_KRLX_476088s_F5.nc,d9b389ca9b3dc6d111c977807491ac7d5fe161ab90754b...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"
4,train,TOR,train/2013/TOR_131031_150019_KLCH_480352_W9.nc,57236e1b75416e47c28ea0c682e3bd8b2705a239db2bec...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"
5,train,WRN,train/2013/WRN_130908_005945_KUDX_1073301n_E2.nc,b3c9e0cf68849493b05e548a55b9a12ba43123201e7006...,NETCDF4,NETCDF4,"{'sweep': 2, 'time': 4, 'lims': 2, 'azimuth': ...","[DBZ, KDP, RHOHV, VEL, WIDTH, ZDR, azimuth_lim...","[azimuth, range, time]"


,variable,representative_files,decoded_shapes,decoded_dtypes,raw_dtypes,fill_values,least_significant_digits,native_chunking,native_filters,native_endianness
0,DBZ,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[1],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
1,KDP,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[2],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
2,RHOHV,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[3],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
3,VEL,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[1],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
4,WIDTH,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[2],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
5,ZDR,6,"[[4, 120, 240, 2]]","[""float32""]","[""float32""]",[-999.0],[3],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
6,frame_labels,6,[[4]],"[""uint8""]","[""uint8""]",[null],[null],"[""contiguous""]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"
7,range_folded_mask,6,"[[4, 120, 240, 2]]","[""uint8""]","[""uint8""]",[null],[null],"[[4, 120, 240, 2]]","[{""blosc"": false, ""bzip2"": false, ""complevel"":...","[""little""]"


variable,split,category,DBZ,KDP,RHOHV,VEL,WIDTH,ZDR
0,test,NUL,0.711706,0.578559,0.706892,0.664605,0.664605,1.000000
1,test,TOR,0.898095,0.829688,0.897396,0.892313,0.892313,1.000000
2,test,WRN,0.719366,0.539575,0.717383,0.672361,0.672361,0.741667
3,train,NUL,0.932674,0.905122,0.932270,0.924253,0.924253,1.000000
4,train,TOR,0.998663,0.944401,0.998663,0.979666,0.979666,1.000000
5,train,WRN,0.682478,0.654280,0.682383,0.661675,0.661675,1.000000


,split,category,archive_member,event_id,episode_id,ef_number,tornado_start_time,tornado_end_time,frame_count,positive_frame_count,frame_labels,frame_times
0,test,NUL,test/2013/NUL_130916_011006_KPDT_472604s_D4.nc,472604,78569,-1.0,,,4,0,"[0, 0, 0, 0]","[2013-09-16T00:55:30, 2013-09-16T01:00:30, 201..."
1,test,TOR,test/2013/TOR_131004_040911_KOAX_472700_A1.nc,472700,78589,0.0,2013-10-04 03:55:00,2013-10-04 04:09:00,4,4,"[1, 1, 1, 1]","[2013-10-04T03:54:30, 2013-10-04T03:59:30, 201..."
2,test,WRN,test/2013/WRN_131005_223742_KPAH_1073330n_H9.nc,476964,79530,-1.0,,,4,0,"[0, 0, 0, 0]","[2013-10-05T22:23:00, 2013-10-05T22:28:00, 201..."
3,train,NUL,train/2013/NUL_131101_063025_KRLX_476088s_F5.nc,476088,79355,-1.0,,,4,0,"[0, 0, 0, 0]","[2013-11-01T06:15:00, 2013-11-01T06:20:00, 201..."
4,train,TOR,train/2013/TOR_131031_150019_KLCH_480352_W9.nc,480352,80194,1.0,2013-10-31 15:00:00,2013-10-31 15:10:00,4,1,"[0, 0, 0, 1]","[2013-10-31T14:45:00, 2013-10-31T14:50:00, 201..."
5,train,WRN,train/2013/WRN_130908_005945_KUDX_1073301n_E2.nc,478070,79074,-1.0,,,4,0,"[0, 0, 0, 0]","[2013-09-08T00:45:00, 2013-09-08T00:50:00, 201..."


Concise audit complete: 6 representative files, 90 file-variable records, 6 frame-label summaries.


## Full 2013 file and frame manifest

This section streams the complete 2013 archive without extracting it permanently. Core logic lives under `src/tornado_detection/data/`.


In [ ]:
import tornado_detection

from tornado_detection.data.manifest import (
    build_archive_manifests,
    validate_manifests,
    write_manifest_artifacts,
)

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


### Build the manifests


In [ ]:
MANIFEST_YEAR = 2013
MANIFEST_ARCHIVE = (
    TORNET_ARCHIVE_DIR
    / f"tornet_{MANIFEST_YEAR}.tar.gz"
)
MANIFEST_WORK_DIR = Path(
    "/content/tornet_manifest_work"
)

if not MANIFEST_ARCHIVE.is_file():
    raise FileNotFoundError(MANIFEST_ARCHIVE)

print("Manifest archive:", MANIFEST_ARCHIVE)
print("Temporary work directory:", MANIFEST_WORK_DIR)


In [ ]:
manifest_result = build_archive_manifests(
    MANIFEST_ARCHIVE,
    expected_year=MANIFEST_YEAR,
    working_directory=MANIFEST_WORK_DIR,
    progress_every=250,
    progress=print,
)

print()
print(
    "File manifest rows:",
    f"{len(manifest_result.file_manifest):,}",
)
print(
    "Frame manifest rows:",
    f"{len(manifest_result.frame_manifest):,}",
)
print(
    "Schema variants:",
    f"{len(manifest_result.schema_summary):,}",
)
print(
    "Build errors:",
    f"{len(manifest_result.errors):,}",
)

display(manifest_result.errors.head(20))
display(manifest_result.schema_summary)


### Validate counts, identities, labels, schemas, and official-split isolation


In [ ]:
EXPECTED_2013_SPLIT_CATEGORY_COUNTS = {
    ("test", "NUL"): 253,
    ("test", "TOR"): 60,
    ("test", "WRN"): 260,
    ("train", "NUL"): 2_548,
    ("train", "TOR"): 329,
    ("train", "WRN"): 621,
}

manifest_validation = validate_manifests(
    manifest_result,
    expected_file_count=4_071,
    expected_frame_count=16_284,
    expected_frames_per_file=4,
    expected_split_category_counts=(
        EXPECTED_2013_SPLIT_CATEGORY_COUNTS
    ),
    expected_dimensions={
        "time": 4,
        "sweep": 2,
        "azimuth": 120,
        "range": 240,
        "lims": 2,
    },
)

display(manifest_validation.checks)
display(
    manifest_validation.category_frame_summary
)

print("Event groups crossing train/test:")
display(manifest_validation.event_split_overlap)

print(
    "Episode groups crossing train/test "
    "(informational):"
)
display(manifest_validation.episode_split_overlap)

manifest_validation.assert_valid()

print(
    "PASS: all required 2013 manifest validations "
    "succeeded"
)


### Write durable Parquet manifests to Google Drive


In [ ]:
MANIFEST_OUTPUT_DIR = (
    TORNET_ARCHIVE_DIR
    / "manifests"
    / "v1"
    / str(MANIFEST_YEAR)
)

manifest_artifacts = write_manifest_artifacts(
    manifest_result,
    manifest_validation,
    MANIFEST_OUTPUT_DIR,
    overwrite=True,
)

print(
    "Manifest artifacts written to:",
    MANIFEST_OUTPUT_DIR,
)

for artifact_name, artifact_path in (
    manifest_artifacts.items()
):
    print(
        f"- {artifact_name}: {artifact_path}"
    )
